## How to run this notebook

These cells are **SQL**, not Python. The `.ipynb` `code` cells are only SQL text.

- **Databricks:** import the notebook, set language to **SQL** (or attach a **SQL warehouse** / cluster that runs SQL). Do not use a Python-only kernel.
- **VS Code / Cursor Jupyter:** if the default kernel is **Python**, cells will fail with `SyntaxError` on `CREATE`/`MERGE` — change the notebook kernel to **SQL** if your setup provides it, or paste the SQL into a Databricks SQL editor / warehouse query.

**Why not temp views in gold?** Staging was split into `CREATE TEMP VIEW` + `MERGE` for readability. The generator now uses **one `MERGE` per table** with an **inline `USING` subquery** (no temp objects). **Materialized views** are a different Databricks feature (managed refresh, often Lakeflow/DBSQL MVs); here we **merge into regular Delta gold tables** for simple, idempotent batch updates.

---


# Bronze — incremental, idempotent load (SQL)

- **Idempotency:** `MERGE` on `_bronze_row_key` = `SHA2(CONCAT_WS(...), 256)` over business columns, matching the prior Python notebook.
- **Prereq:** `retail_medallion_sql/ddl/00_schemas.sql` and `01_bronze_tables.sql` (tables must exist).
- **Source path:** `practice` catalog, volume `demo_dw_raw.raw_data`, subfolders `customers`, `products`, `transactions`.

Run the three **MERGE** cells in order (independent; can be parallelized in a Job if desired).


In [ ]:
-- Customers: merge from read_files
MERGE INTO practice.bronze.customers t
USING (
  SELECT
    CAST(customer_id AS INT)                      AS customer_id,
    first_name,
    last_name,
    email,
    city,
    state,
    CAST(CAST(signup_date AS STRING) AS DATE)     AS signup_date,
    loyalty_tier,
    current_timestamp()                             AS _ingest_ts,
    _metadata.file_path                             AS _source_path,
    _metadata.file_name                              AS _source_name,
    CAST(_metadata.size AS BIGINT)                  AS _source_size_bytes,
    SHA2(
      CONCAT_WS(
        '||',
        COALESCE(CAST(customer_id AS STRING), ''),
        COALESCE(CAST(first_name  AS STRING), ''),
        COALESCE(CAST(last_name   AS STRING), ''),
        COALESCE(CAST(email       AS STRING), ''),
        COALESCE(CAST(city        AS STRING), ''),
        COALESCE(CAST(state       AS STRING), ''),
        COALESCE(CAST(CAST(CAST(signup_date AS STRING) AS DATE) AS STRING), ''),
        COALESCE(CAST(loyalty_tier AS STRING), '')
      ),
      256
    ) AS _bronze_row_key
  FROM read_files(
    '/Volumes/practice/demo_dw_raw/raw_data/customers/',
    format      => 'csv',
    header      => true,
    multiLine   => true,
    inferSchema => true
  )
) s
ON t._bronze_row_key = s._bronze_row_key
WHEN MATCHED THEN UPDATE SET
  t.customer_id = s.customer_id,
  t.first_name = s.first_name,
  t.last_name = s.last_name,
  t.email = s.email,
  t.city = s.city,
  t.state = s.state,
  t.signup_date = s.signup_date,
  t.loyalty_tier = s.loyalty_tier,
  t._ingest_ts = s._ingest_ts,
  t._source_path = s._source_path,
  t._source_name = s._source_name,
  t._source_size_bytes = s._source_size_bytes,
  t._bronze_row_key = s._bronze_row_key
WHEN NOT MATCHED THEN INSERT
  (customer_id, first_name, last_name, email, city, state, signup_date, loyalty_tier, _ingest_ts, _source_path, _source_name, _source_size_bytes, _bronze_row_key)
VALUES
  (s.customer_id, s.first_name, s.last_name, s.email, s.city, s.state, s.signup_date, s.loyalty_tier, s._ingest_ts, s._source_path, s._source_name, s._source_size_bytes, s._bronze_row_key);


In [ ]:
-- Products: merge from read_files
MERGE INTO practice.bronze.products t
USING (
  SELECT
    CAST(product_id AS INT)                       AS product_id,
    product_name,
    category,
    brand,
    CAST(CAST(price AS STRING) AS DOUBLE)         AS price,
    CAST(CAST(created_date AS STRING) AS DATE)    AS created_date,
    current_timestamp()                            AS _ingest_ts,
    _metadata.file_path                            AS _source_path,
    _metadata.file_name                             AS _source_name,
    CAST(_metadata.size AS BIGINT)                 AS _source_size_bytes,
    SHA2(
      CONCAT_WS('||',
        COALESCE(CAST(CAST(product_id AS STRING) AS STRING), ''),
        COALESCE(CAST(product_name AS STRING), ''),
        COALESCE(CAST(category     AS STRING), ''),
        COALESCE(CAST(brand        AS STRING), ''),
        COALESCE(CAST(CAST(CAST(price AS STRING) AS DOUBLE) AS STRING), ''),
        COALESCE(CAST(created_date AS STRING), '')
      ),
      256
    ) AS _bronze_row_key
  FROM read_files(
    '/Volumes/practice/demo_dw_raw/raw_data/products/',
    format      => 'csv',
    header      => true,
    multiLine   => true,
    inferSchema => true
  )
) s
ON t._bronze_row_key = s._bronze_row_key
WHEN MATCHED THEN UPDATE SET
  t.product_id = s.product_id,
  t.product_name = s.product_name,
  t.category = s.category,
  t.brand = s.brand,
  t.price = s.price,
  t.created_date = s.created_date,
  t._ingest_ts = s._ingest_ts,
  t._source_path = s._source_path,
  t._source_name = s._source_name,
  t._source_size_bytes = s._source_size_bytes,
  t._bronze_row_key = s._bronze_row_key
WHEN NOT MATCHED THEN INSERT
  (product_id, product_name, category, brand, price, created_date, _ingest_ts, _source_path, _source_name, _source_size_bytes, _bronze_row_key)
VALUES
  (s.product_id, s.product_name, s.category, s.brand, s.price, s.created_date, s._ingest_ts, s._source_path, s._source_name, s._source_size_bytes, s._bronze_row_key);


In [ ]:
-- Transactions: merge from read_files (empty customer_id → null)
MERGE INTO practice.bronze.transactions t
USING (
  SELECT
    CAST(transaction_id AS INT)  AS transaction_id,
    CASE
      WHEN customer_id IS NULL OR trim(cast(customer_id AS string)) = '' THEN NULL
      ELSE CAST(customer_id AS INT)
    END                            AS customer_id,
    CAST(product_id AS INT)         AS product_id,
    CAST(quantity AS INT)            AS quantity,
    CAST(CAST(transaction_amount AS STRING) AS DOUBLE) AS transaction_amount,
    to_timestamp(CAST(transaction_timestamp AS STRING)) AS transaction_timestamp,
    channel,
    current_timestamp()             AS _ingest_ts,
    _metadata.file_path            AS _source_path,
    _metadata.file_name             AS _source_name,
    CAST(_metadata.size AS BIGINT)  AS _source_size_bytes,
    SHA2(
      CONCAT_WS('||',
        COALESCE(CAST(CAST(transaction_id AS STRING) AS STRING), ''),
        COALESCE(CASE
          WHEN customer_id IS NULL OR trim(cast(customer_id AS string)) = '' THEN 'NULL'
          ELSE CAST(CAST(customer_id AS INT) AS STRING)
        END, ''),
        COALESCE(CAST(CAST(product_id AS STRING) AS STRING), ''),
        COALESCE(CAST(CAST(quantity AS STRING) AS STRING), ''),
        COALESCE(CAST(CAST(transaction_amount AS STRING) AS STRING), ''),
        COALESCE(CAST(transaction_timestamp AS STRING), ''),
        COALESCE(CAST(channel AS STRING), '')
      ),
      256
    ) AS _bronze_row_key
  FROM read_files(
    '/Volumes/practice/demo_dw_raw/raw_data/transactions/',
    format      => 'csv',
    header      => true,
    multiLine   => true,
    inferSchema => true
  )
) s
ON t._bronze_row_key = s._bronze_row_key
WHEN MATCHED THEN UPDATE SET
  t.transaction_id = s.transaction_id,
  t.customer_id = s.customer_id,
  t.product_id = s.product_id,
  t.quantity = s.quantity,
  t.transaction_amount = s.transaction_amount,
  t.transaction_timestamp = s.transaction_timestamp,
  t.channel = s.channel,
  t._ingest_ts = s._ingest_ts,
  t._source_path = s._source_path,
  t._source_name = s._source_name,
  t._source_size_bytes = s._source_size_bytes,
  t._bronze_row_key = s._bronze_row_key
WHEN NOT MATCHED THEN INSERT
  (transaction_id, customer_id, product_id, quantity, transaction_amount, transaction_timestamp, channel, _ingest_ts, _source_path, _source_name, _source_size_bytes, _bronze_row_key)
VALUES
  (s.transaction_id, s.customer_id, s.product_id, s.quantity, s.transaction_amount, s.transaction_timestamp, s.channel, s._ingest_ts, s._source_path, s._source_name, s._source_size_bytes, s._bronze_row_key);


In [ ]:
-- Quick validation
SELECT
  (SELECT COUNT(*) FROM practice.bronze.customers)     AS n_customers,
  (SELECT COUNT(*) FROM practice.bronze.products)   AS n_products,
  (SELECT COUNT(*) FROM practice.bronze.transactions)  AS n_transactions;


### Optional: layer checkpoint

If you created `practice.meta.layer_run_state` (see `retail_medallion_sql/ddl/04_pipeline_control.sql`), run:

```sql
INSERT INTO practice.meta.layer_run_state (layer, last_event_ts, run_id, run_notes)
VALUES ('bronze', current_timestamp(), '01_bronze_incremental', 'ok');
```

Otherwise skip.
